# 02 - Eksperimen & Komparasi 5 Model (*From Scratch* vs `scikit-learn`)
**Penanggung Jawab:** Kolektif Seluruh Anggota (Anggota 1 - 5)  
**Kelompok:** G15 - MessiLearning (IF3170 Inteligensi Artifisial)

---
Notebook ini berisi pengujian 5 algoritma Supervised Learning *from scratch* menggunakan data sintetis (selama fase paralel) maupun data bersih hasil `DataPipeline`, lengkap dengan komparasi metrik (Akurasi, Presisi, Recall, F1-Score, Waktu Eksekusi) terhadap `scikit-learn`.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.utils import (
    set_global_seed,
    RANDOM_SEED,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    calculate_custom_cost,
    k_fold_split,
)
from src.dummy_generator import generate_clean_synthetic_data, generate_dtl_synthetic_data
from src.knn import KNN
from src.dtl import DecisionTree, export_tree_visualization
from src.logistic_regression import LogisticRegression, generate_loss_contour_video
from src.svm import SVM, plot_svm_margin_2d
from src.naive_bayes import NaiveBayes, export_bayesian_network_graph

set_global_seed(RANDOM_SEED)
X_dummy, y_dummy = generate_clean_synthetic_data(n_samples=500, n_features=6)
X_dtl_dummy, y_dtl_dummy = generate_dtl_synthetic_data(n_samples=500)
print(f"Dummy Clean Shape: {X_dummy.shape}, Dummy DTL Shape: {X_dtl_dummy.shape}")

## 1. K-Nearest Neighbors (KNN) — Anggota 1
Komparasi `src.knn.KNN` vs `sklearn.neighbors.KNeighborsClassifier`

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# TODO (Anggota 1): Evaluasi KNN from scratch vs scikit-learn

## 2. Decision Tree Learning (DTL — Varian Pilihan: C4.5) — Anggota 2 (`13524053`)

Sesuai Cakupan Poin 2 spesifikasi (*"Pilih salah satu algoritma DTL: ID3, C4.5, atau CART"*), kelompok memilih **C4.5** sebagai varian utama DTL *from scratch* pada `src/dtl.py` (100% pure NumPy, bebas `sklearn`, dan tetap berupa **1 pohon keputusan tunggal / non-ensemble**):
1. **Spesifikasi Wajib (C4.5)**: Menggunakan *Gain Ratio* dengan heuristik *above-average gain* [2], penanganan fitur numerik kontinu (*midpoint threshold*), kategorikal *multi-way*, penanganan *missing values* (`NaN`) berbobot pecahan ala Quinlan [2], *save/load*, serta komparasi terhadap `sklearn.tree.DecisionTreeClassifier(criterion='entropy', ccp_alpha=...)` pada **dataset penginapan resmi hasil pipeline Anggota 1** (`train_dtl.csv` / `val_dtl.csv` vs `train_clean.csv` / `val_clean.csv`).
2. **Bonus Visualisasi Tree (+1 Poin)**: Fungsi `export_tree_visualization(model, top_n=3, ...)` yang menggambar struktur percabangan pohon dari *root* hingga kedalaman `top_n` ke `output/plots/dtl_tree_top_n.png`.
3. **Bonus Optimasi Model DTL (+2 Poin — Di Luar Salindia Materi 10a/10c)**:
   - **Opsi A — Particle Swarm Optimization (PSO)** (*Kennedy & Eberhart, 1995 [7]; Shi & Eberhart, 1998 [8]*): Mengoptimasi struktur & *hyperparameter* C4.5 (`max_depth`, `min_samples_leaf`, `min_samples_split`, `log10(ccp_alpha)`) secara deterministik terhadap *fitness* **Macro F1** validasi internal di dalam `train_dtl` (bebas *leakage* terhadap `val_dtl`), lalu menghasilkan **1 model C4.5 tunggal terbaik**.
   - **Opsi B — Histogram-Based Quantile Split Optimization** (*Ke et al., 2017 [5]; Chen & Guestrin, 2016 [6]*) + **Minimal Cost-Complexity Weakest-Link Pruning Path** (*Breiman et al., 1984 [4], Bab 3*): Pra-diskretisasi kuantil $B=64$ *bins* (bebas *leakage*, $O(B)$ per simpul) + pemangkasan *weakest-link* berbasis *weighted impurity* $R_\alpha(T) = R(T) + \alpha|L(T)|$ tanpa membuang data latih seperti REP (Materi 10c).
   - **Opsi C — Surrogate Splits** (*Breiman et al., 1984 [4], Bab 5.3*): Diuji pada studi ablasi sebagai pembanding penanganan `NaN` berbasis asosiasi prediktif $\lambda(s^* \mid s') > 0$.

**Referensi Ilmiah (Format IEEE):**
- `[1]` J. R. Quinlan, "Induction of decision trees," *Machine Learning*, vol. 1, no. 1, pp. 81–106, 1986.
- `[2]` J. R. Quinlan, *C4.5: Programs for Machine Learning*. San Mateo, CA, USA: Morgan Kaufmann, 1993.
- `[3]` T. M. Mitchell, *Machine Learning*. New York, NY, USA: McGraw-Hill, 1997, ch. 3.
- `[4]` L. Breiman, J. H. Friedman, R. A. Olshen, and C. J. Stone, *Classification and Regression Trees*. Wadsworth, 1984, ch. 3 & ch. 5.
- `[5]` G. Ke *et al.*, "LightGBM: A highly efficient gradient boosting decision tree," in *Advances in Neural Information Processing Systems (NeurIPS)*, vol. 30, 2017, pp. 3146–3154.
- `[6]` T. Chen and C. Guestrin, "XGBoost: A scalable tree boosting system," in *Proc. 22nd ACM SIGKDD Int. Conf. Knowledge Discovery and Data Mining (KDD)*, 2016, pp. 785–794.
- `[7]` J. Kennedy and R. Eberhart, "Particle swarm optimization," in *Proc. IEEE Int. Conf. Neural Networks (ICNN)*, vol. 4, 1995, pp. 1942–1948.
- `[8]` Y. Shi and R. Eberhart, "A modified particle swarm optimizer," in *Proc. IEEE Int. Conf. Evolutionary Computation (ICEC)*, 1998, pp. 69–73.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import chi2
from src.dtl import DecisionTree, PSODecisionTreeOptimizer, export_tree_visualization
from src.utils import confusion_matrix

root_dir = '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'
train_dtl_path = os.path.join(root_dir, 'data', 'processed', 'train_dtl.csv')
val_dtl_path = os.path.join(root_dir, 'data', 'processed', 'val_dtl.csv')
train_clean_path = os.path.join(root_dir, 'data', 'processed', 'train_clean.csv')
val_clean_path = os.path.join(root_dir, 'data', 'processed', 'val_clean.csv')
raw_train_path = os.path.join(root_dir, 'data', 'raw', 'train.csv')

def macro_f1_score(y_true, y_pred):
    return 0.5 * (f1_score(y_true, y_pred) + f1_score(1 - y_true, 1 - y_pred))

def mcnemar_test(y_true, pred_a, pred_b):
    """Uji signifikansi McNemar (Materi 13) dengan koreksi kontinuitas Edwards (df=1)."""
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    stat = ((abs(b - c) - 1.0) ** 2) / (b + c) if (b + c) > 0 else 0.0
    p_val = float(1.0 - chi2.cdf(stat, df=1)) if (b + c) > 0 else 1.0
    return b, c, stat, p_val

# 1. Muat Dataset Resmi Hasil Pipeline Anggota 1 (train_dtl.csv + val_dtl.csv)
has_official_split = all(os.path.exists(p) for p in (train_dtl_path, val_dtl_path, train_clean_path, val_clean_path))
if has_official_split:
    df_tr_dtl = pd.read_csv(train_dtl_path, dtype={'id': str})
    df_va_dtl = pd.read_csv(val_dtl_path, dtype={'id': str})
    df_tr_cln = pd.read_csv(train_clean_path, dtype={'id': str})
    df_va_cln = pd.read_csv(val_clean_path, dtype={'id': str})
    assert (df_tr_dtl['id'] == df_tr_cln['id']).all(), 'ID train_dtl != train_clean!'
    assert (df_va_dtl['id'] == df_va_cln['id']).all(), 'ID val_dtl != val_clean!'
    src_label = 'data/processed/train_dtl.csv + val_dtl.csv (Pipeline Resmi Anggota 1)'
elif os.path.exists(raw_train_path):
    df_full = pd.read_csv(raw_train_path, dtype={'id': str})
    n_val = int(0.2 * len(df_full))
    df_tr_dtl, df_va_dtl = df_full.iloc[:-n_val].copy(), df_full.iloc[-n_val:].copy()
    df_tr_cln, df_va_cln = None, None
    src_label = raw_train_path
else:
    X_syn, y_syn = generate_dtl_synthetic_data(n_samples=600, random_state=RANDOM_SEED)
    df_full = pd.DataFrame(X_syn, columns=['lead_time', 'adr_price', 'market_segment', 'deposit_type'])
    df_full['is_canceled'] = y_syn
    df_tr_dtl, df_va_dtl = df_full.iloc[:480].copy(), df_full.iloc[480:].copy()
    df_tr_cln, df_va_cln = None, None
    src_label = 'synthetic_fallback'

target_col = 'is_canceled' if 'is_canceled' in df_tr_dtl.columns else df_tr_dtl.columns[-1]
dtl_feat_names = [c for c in df_tr_dtl.columns if c not in ('id', target_col)]
cat_col_names = {'meal_package', 'room_category', 'booking_channel', 'market_segment', 'deposit_type'}
dtl_cat_indices = [i for i, c in enumerate(dtl_feat_names) if c in cat_col_names or df_tr_dtl[c].dtype == object or str(df_tr_dtl[c].dtype) in ('str', 'string')]

X_tr_dtl = df_tr_dtl[dtl_feat_names].to_numpy(dtype=object)
y_tr_dtl = df_tr_dtl[target_col].to_numpy(dtype=int)
X_va_dtl = df_va_dtl[dtl_feat_names].to_numpy(dtype=object)
y_va_dtl = df_va_dtl[target_col].to_numpy(dtype=int)

base_cols = dtl_feat_names[:17] if len(dtl_feat_names) >= 17 else dtl_feat_names
has_nan_base = df_va_dtl[base_cols].isna().any(axis=1).to_numpy()

print(f"[DTL-C4.5] Sumber data : {src_label}")
print(f"[DTL-C4.5] Train shape : {X_tr_dtl.shape} | Val shape: {X_va_dtl.shape}")
print(f"[DTL-C4.5] Fitur kat.  : {[dtl_feat_names[i] for i in dtl_cat_indices]}")

# =============================================================================
# EKSPERIMEN 1: ABLATION STUDY BONUS 6 PADA VARIAN C4.5 (Baseline Kuliah vs Opsi A, B, C)
# =============================================================================
c45_ablation = [
    ('Baseline Kuliah: C4.5 Exact (Unpruned)', lambda: DecisionTree(criterion='c45', split_strategy='exact', missing_strategy='fractional', max_depth=18, min_samples_leaf=5, use_pruning=False, categorical_features=dtl_cat_indices, random_state=RANDOM_SEED)),
    ('Baseline Kuliah: C4.5 Exact + REP (Materi 10c)', lambda: DecisionTree(criterion='c45', split_strategy='exact', missing_strategy='fractional', max_depth=18, min_samples_leaf=5, use_pruning=True, pruning_method='rep', val_split_ratio=0.15, categorical_features=dtl_cat_indices, random_state=RANDOM_SEED)),
    ('Opsi B: C4.5 Hist(B=64) + Minimal CCP', lambda: DecisionTree(criterion='c45', split_strategy='histogram', max_bins=64, missing_strategy='fractional', max_depth=18, min_samples_leaf=5, use_pruning=True, pruning_method='ccp', ccp_alpha=0.0003, categorical_features=dtl_cat_indices, random_state=RANDOM_SEED)),
    ('Opsi C: C4.5 Hist(B=64) + Surrogate Splits + CCP', lambda: DecisionTree(criterion='c45', split_strategy='histogram', max_bins=64, missing_strategy='surrogate', max_surrogates=3, max_depth=18, min_samples_leaf=5, use_pruning=True, pruning_method='ccp', ccp_alpha=0.0003, categorical_features=dtl_cat_indices, random_state=RANDOM_SEED)),
    ('Opsi A1 (Pemenang): PSO + C4.5 Hist(B=64) + CCP', lambda: PSODecisionTreeOptimizer(criterion='c45', split_strategy='histogram', max_bins=64, missing_strategy='fractional', n_particles=8, n_iters=4, categorical_features=dtl_cat_indices, random_state=RANDOM_SEED)),
]

ablation_rows = []
val_preds_store = {}
pso_winner_opt = None
for label, builder in c45_ablation:
    t0 = time.perf_counter()
    clf = builder()
    clf.fit(X_tr_dtl, y_tr_dtl)
    elapsed = time.perf_counter() - t0
    preds = clf.predict(X_va_dtl)
    val_preds_store[label] = preds
    tree_obj = clf.best_model_ if isinstance(clf, PSODecisionTreeOptimizer) else clf
    if isinstance(clf, PSODecisionTreeOptimizer):
        pso_winner_opt = clf
    ablation_rows.append({
        'Konfigurasi C4.5 (Holdout val_dtl)': label,
        'Val Macro F1': macro_f1_score(y_va_dtl, preds),
        'Macro F1 (NaN)': macro_f1_score(y_va_dtl[has_nan_base], preds[has_nan_base]) if np.any(has_nan_base) else np.nan,
        'Macro F1 (Full)': macro_f1_score(y_va_dtl[~has_nan_base], preds[~has_nan_base]) if np.any(~has_nan_base) else np.nan,
        'Accuracy': accuracy_score(y_va_dtl, preds),
        'Precision': precision_score(y_va_dtl, preds),
        'Recall': recall_score(y_va_dtl, preds),
        'Cost (2FP+FN)': calculate_custom_cost(y_va_dtl, preds),
        'Nodes': tree_obj.get_n_nodes(),
        'Depth': tree_obj.get_depth(),
        'Time (s)': elapsed,
    })

df_c45_ablation = pd.DataFrame(ablation_rows)
print("\n=== TABEL 1: ABLATION STUDY BONUS 6 — VARIAN C4.5 (BASELINE KULIAH VS OPSI A, B, C) ===")
display(df_c45_ablation.round(4))

print(f"[PSO Best Params] {pso_winner_opt.best_params_} | Inner Fitness: {pso_winner_opt.best_score_:.5f}")
print(f"[PSO Convergence] {[(h['iter'], round(h['g_best_macro_f1'], 5)) for h in pso_winner_opt.history_]}")

b_mc, c_mc, stat_mc, p_mc = mcnemar_test(
    y_va_dtl,
    val_preds_store['Baseline Kuliah: C4.5 Exact + REP (Materi 10c)'],
    val_preds_store['Opsi A1 (Pemenang): PSO + C4.5 Hist(B=64) + CCP'],
)
print(f"[McNemar Test — Baseline REP vs Opsi A1 PSO+Hist+CCP] b={b_mc}, c={c_mc}, chi2={stat_mc:.4f}, p-value={p_mc:.6f}")

# =============================================================================
# EKSPERIMEN 2: KOMPARASI HEAD-TO-HEAD C4.5 FROM SCRATCH VS SCIKIT-LEARN (SPEK HAL. 10)
# =============================================================================
if has_official_split:
    cln_feat_names = [c for c in df_tr_cln.columns if c not in ('id', target_col)]
    X_tr_sk = df_tr_cln[cln_feat_names].to_numpy(dtype=float)
    X_va_sk = df_va_cln[cln_feat_names].to_numpy(dtype=float)
else:
    X_tr_sk = np.nan_to_num(pd.DataFrame(X_tr_dtl).apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float), nan=0.0)
    X_va_sk = np.nan_to_num(pd.DataFrame(X_va_dtl).apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float), nan=0.0)

best_c45_model = pso_winner_opt.best_model_
best_c45_model.feature_names_ = dtl_feat_names
best_c45_preds = val_preds_store['Opsi A1 (Pemenang): PSO + C4.5 Hist(B=64) + CCP']

comp_rows = [
    {
        'Komparasi C4.5 (Holdout Resmi 5,462 baris)': 'Scratch C4.5 Baseline Kuliah (Exact + REP)',
        'Val Macro F1': macro_f1_score(y_va_dtl, val_preds_store['Baseline Kuliah: C4.5 Exact + REP (Materi 10c)']),
        'Accuracy': accuracy_score(y_va_dtl, val_preds_store['Baseline Kuliah: C4.5 Exact + REP (Materi 10c)']),
        'Precision': precision_score(y_va_dtl, val_preds_store['Baseline Kuliah: C4.5 Exact + REP (Materi 10c)']),
        'Recall': recall_score(y_va_dtl, val_preds_store['Baseline Kuliah: C4.5 Exact + REP (Materi 10c)']),
        'Cost (2FP+FN)': calculate_custom_cost(y_va_dtl, val_preds_store['Baseline Kuliah: C4.5 Exact + REP (Materi 10c)']),
        'Nodes': int(df_c45_ablation.loc[1, 'Nodes']),
        'Depth': int(df_c45_ablation.loc[1, 'Depth']),
    },
    {
        'Komparasi C4.5 (Holdout Resmi 5,462 baris)': 'Scratch C4.5 Pemenang (PSO + Hist B=64 + CCP)',
        'Val Macro F1': macro_f1_score(y_va_dtl, best_c45_preds),
        'Accuracy': accuracy_score(y_va_dtl, best_c45_preds),
        'Precision': precision_score(y_va_dtl, best_c45_preds),
        'Recall': recall_score(y_va_dtl, best_c45_preds),
        'Cost (2FP+FN)': calculate_custom_cost(y_va_dtl, best_c45_preds),
        'Nodes': best_c45_model.get_n_nodes(),
        'Depth': best_c45_model.get_depth(),
    },
]

for alpha_sk in (round(pso_winner_opt.best_params_['ccp_alpha'], 6), 0.01):
    sk_c45 = DecisionTreeClassifier(criterion='entropy', ccp_alpha=alpha_sk, random_state=RANDOM_SEED)
    sk_c45.fit(X_tr_sk, y_tr_dtl)
    sk_preds = sk_c45.predict(X_va_sk)
    comp_rows.append({
        'Komparasi C4.5 (Holdout Resmi 5,462 baris)': f"Sklearn C4.5 (criterion='entropy', ccp_alpha={alpha_sk})",
        'Val Macro F1': macro_f1_score(y_va_dtl, sk_preds),
        'Accuracy': accuracy_score(y_va_dtl, sk_preds),
        'Precision': precision_score(y_va_dtl, sk_preds),
        'Recall': recall_score(y_va_dtl, sk_preds),
        'Cost (2FP+FN)': calculate_custom_cost(y_va_dtl, sk_preds),
        'Nodes': sk_c45.tree_.node_count,
        'Depth': sk_c45.get_depth(),
    })

df_c45_comp = pd.DataFrame(comp_rows)
print("\n=== TABEL 2: KOMPARASI C4.5 FROM SCRATCH (val_dtl) VS SCIKIT-LEARN (val_clean) ===")
display(df_c45_comp.round(4))

# =============================================================================
# EKSPERIMEN 3: VERIFIKASI STABILITAS 5-FOLD CV & ANTI-OVERFITTING MODEL PEMENANG
# =============================================================================
bp = pso_winner_opt.best_params_
cv_tr_f1, cv_va_f1 = [], []
for tr_idx, va_idx in k_fold_split(X_tr_dtl, y_tr_dtl, k=5, seed=RANDOM_SEED):
    m_cv = DecisionTree(
        criterion='c45', split_strategy='histogram', max_bins=64, missing_strategy='fractional',
        max_depth=bp['max_depth'], min_samples_leaf=bp['min_samples_leaf'],
        min_samples_split=bp['min_samples_split'], use_pruning=True, pruning_method='ccp',
        ccp_alpha=bp['ccp_alpha'], categorical_features=dtl_cat_indices, random_state=RANDOM_SEED,
    )
    m_cv.fit(X_tr_dtl[tr_idx], y_tr_dtl[tr_idx])
    cv_tr_f1.append(macro_f1_score(y_tr_dtl[tr_idx], m_cv.predict(X_tr_dtl[tr_idx])))
    cv_va_f1.append(macro_f1_score(y_tr_dtl[va_idx], m_cv.predict(X_tr_dtl[va_idx])))
print(f"\n[5-Fold CV Stabilitas Pemenang Opsi A1] Train Macro F1: {np.mean(cv_tr_f1):.4f} ± {np.std(cv_tr_f1):.4f} | Val Macro F1: {np.mean(cv_va_f1):.4f} ± {np.std(cv_va_f1):.4f} | Gap: {np.mean(cv_tr_f1)-np.mean(cv_va_f1):.4f}")

# =============================================================================
# EKSPERIMEN 4: SAVE/LOAD VERIFICATION, VISUALISASI TOP-3 (+1 POIN), & ERROR ANALYSIS
# =============================================================================
model_out_path = os.path.join(root_dir, 'output', 'models', 'dtl_model.pkl')
plot_out_path = os.path.join(root_dir, 'output', 'plots', 'dtl_tree_top_n.png')
best_c45_model.save(model_out_path)
loaded_dtl_model = DecisionTree.load(model_out_path)

assert np.allclose(best_c45_model.predict_proba(X_va_dtl), loaded_dtl_model.predict_proba(X_va_dtl))
assert np.array_equal(best_c45_preds, loaded_dtl_model.predict(X_va_dtl))
print(f"[Save/Load Verified] Prediksi & probabilitas 100% identik pada {len(X_va_dtl):,} baris val_dtl.")

export_tree_visualization(loaded_dtl_model, top_n=3, feature_names=dtl_feat_names, output_path=plot_out_path)

tn, fp, fn, tp = confusion_matrix(y_va_dtl, best_c45_preds)
print(f"\n=== ERROR ANALYSIS MODEL C4.5 PEMENANG (val_dtl: {len(X_va_dtl):,} BARIS) ===")
print(f"Confusion Matrix : TN={tn:,}, FP={fp:,}, FN={fn:,}, TP={tp:,}")
print(f"Val Macro F1     : {macro_f1_score(y_va_dtl, best_c45_preds):.4f} | Accuracy: {accuracy_score(y_va_dtl, best_c45_preds):.4f} | Cost: {calculate_custom_cost(y_va_dtl, best_c45_preds):.0f}")
print(f"  -> Baris mengandung NaN ({has_nan_base.sum():,} baris): Macro F1 = {macro_f1_score(y_va_dtl[has_nan_base], best_c45_preds[has_nan_base]):.4f}")
print(f"  -> Baris lengkap tanpa NaN ({(~has_nan_base).sum():,} baris): Macro F1 = {macro_f1_score(y_va_dtl[~has_nan_base], best_c45_preds[~has_nan_base]):.4f}")

df_imp = pd.DataFrame({'Feature': dtl_feat_names, 'Importance': loaded_dtl_model.feature_importances_}).sort_values('Importance', ascending=False)
print("\nTop-10 Fitur Paling Berpengaruh (Feature Importances C4.5):")
display(df_imp.head(10).round(4))

## 3. Logistic Regression — Anggota 3
Komparasi `src.logistic_regression.LogisticRegression` ($L_1, L_2$, Elastic Net + Adam) vs `sklearn.linear_model.LogisticRegression` & Video Lintasan Loss

In [ ]:
from sklearn.linear_model import LogisticRegression as SklearnLogReg
# TODO (Anggota 3): Evaluasi LogisticRegression from scratch vs scikit-learn & render video kontur

## 4. Support Vector Machine (SVM) — Anggota 4
Komparasi `src.svm.SVM` (Linear, Polynomial, Gaussian RBF) vs `sklearn.svm.SVC` & Plot Bidang Pemisah 2D

In [ ]:
from sklearn.svm import SVC
# TODO (Anggota 4): Evaluasi SVM from scratch vs scikit-learn & simpan 3 plot margin 2D

## 5. Naive Bayes & TAN — Anggota 5
Komparasi `src.naive_bayes.NaiveBayes` vs `sklearn.naive_bayes.GaussianNB` & Visualisasi Graf DAG Bayesian Network

In [ ]:
from sklearn.naive_bayes import GaussianNB
# TODO (Anggota 5): Evaluasi NaiveBayes / TAN from scratch vs scikit-learn & simpan graf BN